# Tugas Praktikum - Wisconsin Breast Cancer

**Nama:** Agus

**NIM:** 244107020049

### Deskripsi Tugas

1. Pisahkan variabel yang dapat digunakan dan variabel yang tidak dapat digunakan.
2. Lakukan encoding pada kolom `diagnosis`.
3. Lakukan standardisasi pada semua kolom numerik.
4. Lakukan stratified split data latih dan data uji dengan rasio 80:20.

### Data

In [1]:
!pip install -q pandas numpy scikit-learn

### Langkah 1 - Memuat dan Memisahkan Variabel

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pathlib import Path

DATA_DIR = Path('data')
TITANIC_PATH = DATA_DIR / 'Titanic-Dataset.csv'
assert TITANIC_PATH.exists(), f'Dataset tidak ditemukan: {TITANIC_PATH}'
WBC_PATH = DATA_DIR / 'wbc.csv'
assert WBC_PATH.exists(), f'Dataset tidak ditemukan: {WBC_PATH}'
wbc_raw = pd.read_csv(WBC_PATH)
all_missing_cols = wbc_raw.columns[wbc_raw.isna().all()].tolist()
unusable_cols = [col for col in ['id', *all_missing_cols] if col in wbc_raw.columns]
wbc = wbc_raw.drop(columns=unusable_cols).copy()
feature_cols = [col for col in wbc.columns if col != 'diagnosis']
numeric_features = wbc[feature_cols].select_dtypes(include=np.number).columns.tolist()
print('Variabel tidak digunakan:', unusable_cols)
print('Jumlah fitur numerik:', len(numeric_features))
display(wbc.head())

Variabel tidak digunakan: ['id', 'Unnamed: 32']
Jumlah fitur numerik: 30


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### Langkah 2 - Encoding Diagnosis

In [3]:
y = wbc['diagnosis'].map({'B': 0, 'M': 1}).rename('diagnosis')
assert y.notna().all()
print('Encoding diagnosis: B = 0, M = 1')
display(y.value_counts().sort_index().to_frame('jumlah'))

Encoding diagnosis: B = 0, M = 1


,jumlah
diagnosis,
0,357
1,212


### Langkah 3 - Stratified Split

In [4]:
X = wbc[numeric_features]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
summary = pd.DataFrame({
    'dataset': ['train', 'test'],
    'baris': [len(X_train), len(X_test)],
    'proporsi malignant': [y_train.mean(), y_test.mean()]
})
display(summary.round(3))

,dataset,baris,proporsi malignant
0,train,455,0.374
1,test,114,0.368


### Langkah 4 - Standardisasi Fitur Numerik

In [5]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=numeric_features, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=numeric_features, index=X_test.index
)
display(X_train_scaled.head())
display(X_train_scaled.iloc[:, :6].agg(['mean', 'std']).round(3))

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
381,-0.862083,-1.009094,-0.861347,-0.786765,-1.158494,-0.615198,-0.654171,-0.707010,0.692286,-0.047962,...,-0.857701,-0.802311,-0.809130,-0.753108,-0.996486,-0.346704,-0.553743,-0.697964,0.481210,-0.604967
144,-0.944093,-0.999941,-0.959931,-0.835771,-1.296173,-0.978211,-0.811038,-1.031182,-1.503224,-0.836375,...,-0.886863,-0.820201,-0.866969,-0.763508,-1.079789,-0.830368,-0.824462,-1.197365,-0.972434,-0.886765
136,-0.672612,-0.610936,-0.695675,-0.643408,0.632048,-0.799324,-0.648599,-0.574468,-1.721321,-0.473253,...,-0.599404,-0.046036,-0.617429,-0.577535,-0.224836,-0.954630,-0.791414,-0.665977,-1.920043,-0.574502
116,-1.453119,-0.819168,-1.349362,-1.145863,-0.111705,0.386082,0.038345,-0.648046,-1.844910,1.247692,...,-1.415120,-1.413837,-1.297775,-1.065297,-0.628198,-0.412339,-0.557962,-1.132642,-2.016737,-0.368322
567,1.841410,2.286005,1.978793,1.726355,1.533350,3.243407,3.172896,2.600480,2.124457,1.039991,...,1.985640,2.221163,2.317423,1.668781,1.436843,3.922159,3.118660,2.253595,1.907458,2.176563


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean
mean,-0.000,0.000,-0.000,-0.000,0.000,-0.000
std,1.001,1.001,1.001,1.001,1.001,1.001
